In [3]:
import tkinter as tk
import random

TRANG_THAI_BAN_DAU = [
    [1, 2, 3],
    [4, 5, 6],
    [0, 7, 8]
]

KET_QUA_DICH = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

def lam_phang_ma_tran(ma_tran):
    mảng_phẳng = []
    for hang in ma_tran:
        mảng_phẳng.extend(hang)
    return mảng_phẳng

TRANG_THAI_DICH_PHANG = tuple(lam_phang_ma_tran(KET_QUA_DICH))
TRANG_THAI_DAU_PHANG = lam_phang_ma_tran(TRANG_THAI_BAN_DAU)

CAC_HUONG = ["Trai", "Phai", "Len", "Xuong"]
HUONG_NGUOC = {"Trai": "Phai", "Phai": "Trai", "Len": "Xuong", "Xuong": "Len"}

class GiaoDienSimpleHillClimbing:
    def __init__(self, cua_so):
        self.cua_so = cua_so
        self.cua_so.title("8-Puzzle - Simple Hill Climbing (Số ô sai)")

        self.ban_co = list(TRANG_THAI_DAU_PHANG)
        self.so_buoc = 0
        self.dang_tu_dong = False
        self._after_id = None

        khung_tren = tk.Frame(cua_so)
        khung_tren.pack(pady=10, fill="x")

        self.nhan_trang_thai = tk.Label(
            khung_tren,
            text="Số bước: 0",
            anchor="w",
            font=("Arial", 11)
        )
        self.nhan_trang_thai.pack(side="left", padx=10, fill="x", expand=True)

        tk.Button(khung_tren, text="Simple Hill Climbing", command=self.giai_thuat_toan, bg="#98fb98", font=("Arial", 10, "bold")).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Dừng Auto", command=self.dung_tu_dong).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Trộn Bàn Cờ", command=lambda: self.tron(10)).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Đặt Lại", command=self.dat_lai).pack(side="right", padx=5)

        khung_chinh = tk.Frame(cua_so)
        khung_chinh.pack(padx=10, pady=10, fill="both", expand=True)

        khung_ban_co = tk.Frame(khung_chinh, bg="darkgoldenrod2", padx=8, pady=8)
        khung_ban_co.pack(side="left", anchor="n")

        self.danh_sach_o = []
        for hang in range(3):
            for cot in range(3):
                o = tk.Label(
                    khung_ban_co,
                    text="",
                    width=4,
                    height=2,
                    font=("Arial", 22, "bold"),
                    bg="white",
                    relief="raised",
                    borderwidth=3
                )
                o.grid(row=hang, column=cot, padx=4, pady=4)
                self.danh_sach_o.append(o)

        khung_dich = tk.Frame(khung_chinh, bg="#888", padx=10, pady=10)
        khung_dich.pack(side="left", padx=20, anchor="n")

        tk.Label(khung_dich, text="GOAL STATE", font=("Arial", 12, "bold"), bg="#888", fg="white").pack(pady=5)

        luoi_dich = tk.Frame(khung_dich, bg="#888")
        luoi_dich.pack()

        for i, gia_tri in enumerate(TRANG_THAI_DICH_PHANG):
            hang, cot = divmod(i, 3)
            chu = "" if gia_tri == 0 else str(gia_tri)
            mau_nen = "#bbb" if gia_tri != 0 else "#999"

            o_dich = tk.Label(
                luoi_dich,
                text=chu,
                width=4,
                height=2,
                font=("Arial", 16, "bold"),
                bg=mau_nen,
                relief="ridge",
                borderwidth=2
            )
            o_dich.grid(row=hang, column=cot, padx=3, pady=3)

        cua_so.bind("<Left>",  lambda e: self.xu_ly_phim("Trai"))
        cua_so.bind("<Right>", lambda e: self.xu_ly_phim("Phai"))
        cua_so.bind("<Up>",    lambda e: self.xu_ly_phim("Len"))
        cua_so.bind("<Down>",  lambda e: self.xu_ly_phim("Xuong"))
        self.ve_giao_dien()

    def _dem_so_o_sai(self, trang_thai):
        o_sai = 0
        for i in range(9):
            if trang_thai[i] != 0 and trang_thai[i] != TRANG_THAI_DICH_PHANG[i]:
                o_sai += 1
        return o_sai

    def ve_giao_dien(self):
        for i, gia_tri in enumerate(self.ban_co):
            o = self.danh_sach_o[i]
            if gia_tri == 0:
                o.config(text="", bg="#555", relief="sunken")
            else:
                o.config(text=str(gia_tri), bg="#f2f2f2", relief="raised")

        if tuple(self.ban_co) == TRANG_THAI_DICH_PHANG:
            self.nhan_trang_thai.config(text=f"Chúc mừng! Đã đạt đích. Số bước: {self.so_buoc}", fg="green")
        else:
            o_sai = self._dem_so_o_sai(tuple(self.ban_co))
            self.nhan_trang_thai.config(text=f"Số bước hiện tại: {self.so_buoc} | Số ô sai (Cost h): {o_sai}", fg="black")

    def dat_lai(self):
        self.dung_tu_dong()
        self.ban_co = list(TRANG_THAI_DAU_PHANG)
        self.so_buoc = 0
        self.ve_giao_dien()

    def xu_ly_phim(self, huong):
        if self.dang_tu_dong or tuple(self.ban_co) == TRANG_THAI_DICH_PHANG:
            return
        if self.di_chuyen_o_trong(self.ban_co, huong):
            self.so_buoc += 1
            self.ve_giao_dien()

    def di_chuyen_o_trong(self, mang_ban_co, huong):
        vi_tri_trong = mang_ban_co.index(0)
        hang, cot = divmod(vi_tri_trong, 3)

        if huong == "Trai": hang_moi, cot_moi = hang, cot - 1
        elif huong == "Phai": hang_moi, cot_moi = hang, cot + 1
        elif huong == "Len": hang_moi, cot_moi = hang - 1, cot
        elif huong == "Xuong": hang_moi, cot_moi = hang + 1, cot
        else: return False

        if not (0 <= hang_moi < 3 and 0 <= cot_moi < 3):
            return False

        vi_tri_moi = hang_moi * 3 + cot_moi
        mang_ban_co[vi_tri_trong], mang_ban_co[vi_tri_moi] = mang_ban_co[vi_tri_moi], mang_ban_co[vi_tri_trong]
        return True

    def tron(self, so_buoc_tron=10):
        self.dung_tu_dong()
        self.ban_co = list(TRANG_THAI_DICH_PHANG)
        self.so_buoc = 0
        huong_truoc = None
        cac_huong = ["Trai", "Phai", "Len", "Xuong"]
        
        for _ in range(so_buoc_tron):
            random.shuffle(cac_huong)
            for huong in cac_huong:
                if huong_truoc and huong == HUONG_NGUOC[huong_truoc]:
                    continue
                if self.di_chuyen_o_trong(self.ban_co, huong):
                    huong_truoc = huong
                    break
        if tuple(self.ban_co) == TRANG_THAI_DICH_PHANG:
            self.tron(so_buoc_tron)
        self.ve_giao_dien()

    def _hang_xom(self, trang_thai):
        idx0 = trang_thai.index(0)
        r, c = divmod(idx0, 3)

        def swap(t, i, j):
            lst = list(t)
            lst[i], lst[j] = lst[j], lst[i]
            return tuple(lst)

        ket_qua = []
        for huong in CAC_HUONG:
            if huong == "Trai": r2, c2 = r, c - 1
            elif huong == "Phai": r2, c2 = r, c + 1
            elif huong == "Len": r2, c2 = r - 1, c
            else: r2, c2 = r + 1, c

            if 0 <= r2 < 3 and 0 <= c2 < 3:
                idx2 = r2 * 3 + c2
                ket_qua.append((swap(trang_thai, idx0, idx2), huong))
        return ket_qua

    def thuat_toan_simple_hill_climbing(self, bat_dau):
        duong_di = []
        trang_thai_hien_tai = bat_dau
        cost_hien_tai = self._dem_so_o_sai(trang_thai_hien_tai)

        while True:
            if trang_thai_hien_tai == TRANG_THAI_DICH_PHANG:
                return duong_di, "THANH_CONG"

            tim_thay_buoc_tot_hon = False
            
            for trang_thai_con, huong in self._hang_xom(trang_thai_hien_tai):
                cost_con = self._dem_so_o_sai(trang_thai_con)
                
                if cost_con < cost_hien_tai:
                    trang_thai_hien_tai = trang_thai_con
                    cost_hien_tai = cost_con
                    duong_di.append(huong)
                    tim_thay_buoc_tot_hon = True
                    break 
            if not tim_thay_buoc_tot_hon:
                return duong_di, "KET_CUC_BO"

    def chay_loi_giai(self, danh_sach_huong, ket_qua_tim, delay_ms=300):
        self.dang_tu_dong = True
        def step(i):
            if not self.dang_tu_dong: return
            if i >= len(danh_sach_huong):
                self.dang_tu_dong = False
                self._after_id = None
                self.ve_giao_dien()
                if ket_qua_tim == "KET_CUC_BO":
                    self.nhan_trang_thai.config(text=f"Dừng: Bị kẹt đỉnh cục bộ (Số ô sai: {self._dem_so_o_sai(tuple(self.ban_co))})", fg="red")
                return

            self.di_chuyen_o_trong(self.ban_co, danh_sach_huong[i])
            self.so_buoc += 1
            self.ve_giao_dien()
            self._after_id = self.cua_so.after(delay_ms, lambda: step(i + 1))
        step(0)

    def dung_tu_dong(self):
        self.dang_tu_dong = False
        if self._after_id is not None:
            try: self.cua_so.after_cancel(self._after_id)
            except Exception: pass
            self._after_id = None

    def giai_thuat_toan(self):
        if self.dang_tu_dong: return
        bat_dau = tuple(self.ban_co)
        self.nhan_trang_thai.config(text="Đang tìm hướng đi tiến bộ đầu tiên...", fg="blue")
        self.cua_so.update_idletasks()

        path, ket_qua_tim = self.thuat_toan_simple_hill_climbing(bat_dau)
        
        if len(path) == 0 and ket_qua_tim == "KET_CUC_BO":
            self.nhan_trang_thai.config(text="Bị kẹt đỉnh cục bộ ngay từ trạng thái đầu!", fg="red")
            return

        self.chay_loi_giai(path, ket_qua_tim)

if __name__ == "__main__":
    cua_so_chinh = tk.Tk()
    GiaoDienSimpleHillClimbing(cua_so_chinh)
    cua_so_chinh.mainloop()